# 05 - Troubleshooting, Monitoring, and Optimization

This chapter focuses on post-deployment reliability: diagnose failures, monitor health, and optimize startup and latency.


## Section 1: Troubleshooting Framework

### Definition
Troubleshooting is systematic process to detect, isolate, and fix production issues.

### Theory
Use layered diagnosis: infrastructure -> dependencies -> app startup -> inference runtime -> user input.

### Motivation
Random debugging wastes time. Structured triage reduces MTTR (mean time to resolution).

### Real-World Example
User reports blank page. Root cause: import error in deployed environment due missing pinned dependency.

### Best Practices
- Keep reproducible runbook.
- Capture failing input and stack trace.
- Validate fix with regression test.

### Common Mistakes
- Hotfixing symptoms instead of root cause.
- No postmortem documentation.


In [ ]:
import pandas as pd

failure_matrix = pd.DataFrame(
    [
        ["Missing package", "ModuleNotFoundError", "Add package + pin version"],
        ["Startup timeout", "App never reaches healthy state", "Cache resources and reduce cold-start work"],
        ["HF token missing", "401 or fallback-only mode", "Set HF_API_TOKEN in secrets"],
        ["Bad user input", "Validation error", "Improve input validation and UI messaging"],
    ],
    columns=["Failure", "Signal", "Fix Pattern"],
)
failure_matrix


## Section 2: Monitoring Fundamentals

### Definition
Monitoring tracks app health over time using logs, errors, latency, and user feedback.

### Theory
Minimum set: request latency, error rate, fallback rate, and startup time.

### Motivation
Without monitoring, you discover outages only after user complaints.

### Real-World Example
Latency suddenly doubles after dependency upgrade; monitoring catches regression quickly.

### Best Practices
- Emit structured logs.
- Surface metrics in UI/admin panel.
- Track trend, not single datapoint.

### Common Mistakes
- Monitoring only uptime, ignoring model quality and response latency.


In [ ]:
from pathlib import Path
import json
import psutil
import time

process = psutil.Process()
mem_mb = process.memory_info().rss / (1024 ** 2)

metrics = {
    "timestamp": time.time(),
    "memory_rss_mb": round(mem_mb, 2),
    "cpu_percent": process.cpu_percent(interval=0.1),
}

root = Path.cwd()
if not (root / "outputs").exists() and (root.parent / "outputs").exists():
    root = root.parent

out_dir = root / "outputs" / "metrics"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "runtime_snapshot.json"
out_file.write_text(json.dumps(metrics, indent=2))
print(metrics)
print("saved to", out_file)


## Section 3: Production Optimization

### Definition
Optimization improves latency, startup time, and memory efficiency while preserving correctness.

### Theory
Key levers for this app:
- `st.cache_resource` for expensive model clients.
- `st.cache_data` for deterministic repeat computations.
- lightweight fallback for degraded conditions.

### Motivation
Free-tier hosts have strict resource budgets.

### Real-World Example
Cold start drops from 11s to 4s after caching initialization.

### Best Practices
- Measure before and after.
- Optimize bottlenecks first.
- Keep correctness tests around optimized path.

### Common Mistakes
- Premature optimization without baseline.
- Caching mutable objects unsafely.


In [ ]:
import time
from functools import lru_cache

def expensive_init(delay: float = 0.25) -> str:
    time.sleep(delay)
    return "resource-ready"

@lru_cache(maxsize=1)
def cached_expensive_init(delay: float = 0.25) -> str:
    time.sleep(delay)
    return "resource-ready"

t0 = time.perf_counter()
expensive_init()
first_plain = time.perf_counter() - t0

t1 = time.perf_counter()
cached_expensive_init()
first_cached = time.perf_counter() - t1

t2 = time.perf_counter()
cached_expensive_init()
second_cached = time.perf_counter() - t2

{
    "plain_seconds": round(first_plain, 4),
    "cached_first_seconds": round(first_cached, 4),
    "cached_repeat_seconds": round(second_cached, 4),
}
